In [3]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\arjav\DevSource\toc-performance-dashboard")

DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
TOC_DIR = BOUNDARIES_DIR / "tocs"
VILLAGE_DIR = BOUNDARIES_DIR / "villages"
JOBS_DIR = DATA_DIR / "jobs" / "lodes_processed"
PARCELS_DIR = DATA_DIR / "parcels" / "processed"

village_clean_path = BOUNDARIES_DIR / "processed" / "village_clean.gpkg"

village_demographics_path = VILLAGE_DIR / "village_demographics.gpkg"

village_streets_path = VILLAGE_DIR / "village_full_street_miles.gpkg"

village_jobs_path = JOBS_DIR / "village_jobs_2022.gpkg"

village_parcels_path = PARCELS_DIR / "parcels_assessor_village_no_duplicates.gpkg"

village_clean_path, village_demographics_path, village_streets_path, village_jobs_path, village_parcels_path

(WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/village_clean.gpkg'),
 WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/villages/village_demographics.gpkg'),
 WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/villages/village_full_street_miles.gpkg'),
 WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/village_jobs_2022.gpkg'),
 WindowsPath('C:/Users/arjav/DevSource/toc-performance-dashboard/data/parcels/processed/parcels_assessor_village_no_duplicates.gpkg'))

In [4]:
village_clean = gpd.read_file(village_clean_path)

In [5]:
village_demographics = gpd.read_file(village_demographics_path)

In [6]:
village_streets = gpd.read_file(village_streets_path)

In [7]:
village_jobs = gpd.read_file(village_jobs_path)

In [8]:
village_parcels = gpd.read_file(village_parcels_path)

In [11]:
print(village_clean.columns)
print(village_demographics.columns)
print(village_streets.columns)
print(village_jobs.columns)
print(village_parcels.columns)

Index(['OBJECTID', 'NAME', 'geometry'], dtype='object')
Index(['OBJECTID', 'NAME', 'intersect_acres', 'hh_weighted', 'income_weighted',
       'rent_weighted', 'ASNQE001_w', 'ASS8E001_w', 'ASS8E002_w', 'ASS8E003_w',
       'ASS9E002_w', 'ASS9E003_w', 'ASORE001_w', 'ASORE003_w', 'ASORE004_w',
       'ASORE010_w', 'ASORE018_w', 'ASORE019_w', 'ASORE021_w', 'ASORE002_w',
       'median_income_approx', 'median_rent_approx', 'ASNQE001', 'ASS8E001',
       'ASS8E002', 'ASS8E003', 'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003',
       'ASORE004', 'ASORE010', 'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002',
       'pop_per_acre', 'hh_per_acre', 'pct_drive_alone', 'pct_transit',
       'pct_bike', 'pct_walk', 'pct_wfh', 'pct_auto', 'geometry'],
      dtype='object')
Index(['OBJECTID', 'NAME', 'ASNQE001_w', 'ASS8E002_w', 'ASS8E003_w',
       'intersect_acres', 'pop_per_acre', 'hh_per_acre', 'ASS9E002_w',
       'ASS9E003_w', 'median_income_approx', 'median_rent_approx',
       'pct_drive_alone', '

In [13]:
# same as before, the parcel data isn't actually aggregated the way it should be. let's do that.

value_agg = (
    village_parcels
    .groupby("NAME", as_index=False)
    .agg({
        "FULLCASHVALUE": "sum",
        "LANDFULLCASHVALUE": "sum",
        "IMPRFULLCASHVALUE": "sum",
        "LPVAMOUNT": "sum",
        "LPVASSESSEDVALUE": "sum",
        "SQFOOTAGE": "sum",
        "NUMBEROFUNITS": "sum",
    })
)

value_agg.head()

,NAME,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS
0,Ahwatukee Foothills,1.983603e+10,5.383996e+09,1.445203e+10,1.102281e+10,1.197546e+09,9.138694e+08,107.0
1,Alhambra,1.954164e+10,4.451980e+09,1.508966e+10,7.725420e+09,8.894958e+08,4.063308e+08,1129.0
2,Camelback East,4.996990e+10,1.323614e+10,3.673376e+10,2.339305e+10,2.758933e+09,8.580249e+08,1626.0
3,Central City,2.841179e+10,7.045698e+09,2.136610e+10,1.498962e+10,2.152483e+09,4.664189e+08,3701.0
4,Deer Valley,3.655525e+10,9.392208e+09,2.716304e+10,1.748644e+10,2.086664e+09,1.195804e+09,682.0


In [14]:
# and now, data cleaning:

village_base = village_clean[["OBJECTID", "NAME", "geometry"]].copy()

# 2) Demographics: keep the fields we actually care about
dem_cols_keep = [
    "NAME",
    "intersect_acres",
    "median_income_approx",
    "median_rent_approx",
    "ASNQE001",      # total pop
    "ASS8E001",      # total housing units
    "ASS8E002",      # occupied units
    "ASS8E003",      # vacant units
    "ASS9E002",      # owner-occupied
    "ASS9E003",      # renter-occupied
    "ASORE001",      # workers 16+
    "ASORE003",
    "ASORE004",
    "ASORE010",
    "ASORE018",
    "ASORE019",
    "ASORE021",
    "ASORE002",
    "pop_per_acre",
    "hh_per_acre",
    "pct_drive_alone",
    "pct_transit",
    "pct_bike",
    "pct_walk",
    "pct_wfh",
    "pct_auto",
]

village_demographics = village_demographics[dem_cols_keep].copy()

# 3) Streets: just grab the street metrics
streets_cols_keep = [
    "NAME",
    "street_miles_total",
    "street_miles_per_acre",
]
village_streets = village_streets[streets_cols_keep].copy()

# 4) Jobs: jobs + acres
jobs_cols_keep = [
    "NAME",
    "jobs_total",
    "village_acres",
    "jobs_per_acre",
]
village_jobs = village_jobs[jobs_cols_keep].copy()

# 5) Parcels: value + units
parcels_cols_keep = [
    "NAME",
    "FULLCASHVALUE",
    "LANDFULLCASHVALUE",
    "IMPRFULLCASHVALUE",
    "LPVAMOUNT",
    "LPVASSESSEDVALUE",
    "SQFOOTAGE",
    "NUMBEROFUNITS",
]
village_parcels = value_agg[parcels_cols_keep].copy()


In [16]:
village_final = (
    village_base
    .merge(village_demographics, on="NAME", how="left")
    .merge(village_streets, on="NAME", how="left")
    .merge(village_jobs, on="NAME", how="left")
    .merge(village_parcels, on="NAME", how="left")
)
village_final.head()

,OBJECTID,NAME,geometry,intersect_acres,median_income_approx,median_rent_approx,ASNQE001,ASS8E001,ASS8E002,ASS8E003,...,jobs_total,village_acres,jobs_per_acre,FULLCASHVALUE,LANDFULLCASHVALUE,IMPRFULLCASHVALUE,LPVAMOUNT,LPVASSESSEDVALUE,SQFOOTAGE,NUMBEROFUNITS
0,16,Ahwatukee Foothills,"POLYGON ((683945.25 859952.062, 683930.25 8598...",22832.880975,116395.759419,2028.861097,80121.395673,34930.417904,33008.638620,1921.779283,...,31854.214182,22832.880975,1.395103,1.983603e+10,5.383996e+09,1.445203e+10,1.102281e+10,1.197546e+09,9.138694e+08,107.0
1,17,Laveen,"POLYGON ((638589.25 843376, 629247.625 838551....",19579.686598,102147.436370,1774.052390,68987.873610,20401.024379,19775.759721,625.264658,...,38138.709412,19579.686598,1.947871,1.019906e+10,2.939275e+09,7.259786e+09,4.355719e+09,4.859999e+08,7.471402e+08,184.0
2,18,South Mountain,"POLYGON ((683945.25 859952.062, 683530.875 860...",25481.856132,72757.192647,1508.724056,125752.337014,43349.585529,41006.060284,2343.525244,...,60064.332489,25481.856132,2.357141,2.112627e+10,5.686000e+09,1.544027e+10,8.699109e+09,1.115734e+09,9.684476e+08,1710.0
3,19,Estrella,"POLYGON ((607040.694 890232.667, 607046.417 89...",26503.624912,75184.153423,1580.769165,103023.425643,27735.695552,26650.951390,1084.744162,...,20731.840319,26503.624912,0.782227,2.224583e+10,5.216087e+09,1.702974e+10,1.047650e+10,1.481555e+09,1.013909e+09,1300.0
4,20,Central City,"POLYGON ((669683.439 897001.625, 672160.625 89...",13600.592409,52293.256403,1266.640455,58532.832473,27345.355650,24096.557696,3248.797954,...,33900.231265,13600.592409,2.492555,2.841179e+10,7.045698e+09,2.136610e+10,1.498962e+10,2.152483e+09,4.664189e+08,3701.0


In [17]:
# Avoid divide-by-zero: if acres == 0, result will be NaN
village_final["taxable_value_per_acre"] = (
    village_final["LPVASSESSEDVALUE"] / village_final["village_acres"]
)

village_final["full_value_per_acre"] = (
    village_final["FULLCASHVALUE"] / village_final["village_acres"]
)

village_final["land_value_per_acre"] = (
    village_final["LANDFULLCASHVALUE"] / village_final["village_acres"]
)


In [ ]:
village_final = village_final.rename(columns={
    "ASNQE001":   "Total_Pop",
    "ASS8E001":   "Total_Housing_Units",
    "ASS8E002":   "Occupied_Units",
    "ASS8E003":   "Vacant_Units",
    "ASS9E002":   "Owner_Occupied",
    "ASS9E003":   "Renter_Occupied",
    "ASORE001":   "Workers_16_plus",
    "ASORE003":   "Workers_Drive_Alone",
    "ASORE004":   "Workers_Carpool",
    "ASORE010":   "Workers_Transit",
    "ASORE018":   "Workers_Bike",
    "ASORE019":   "Workers_Walk",
    "ASORE021":   "Workers_WFH",
    "ASORE002":   "Workers_Auto_Total",
    "median_income_approx": "Median_Income",
    "median_rent_approx":   "Median_Rent",
    "street_miles_total":   "Street_Miles_Total",
    "street_miles_per_acre":"Street_Miles_per_Acre",
    "jobs_total":           "Jobs_Total",
    "jobs_per_acre":        "Jobs_per_Acre",
})

In [19]:
village_out_path = OUTPUT_DIR / "villages_dashboard.gpkg"
village_final.to_file(village_out_path, driver="GPKG")

print("Saved village dashboard layer:", village_out_path)


Saved village dashboard layer: C:\Users\arjav\DevSource\toc-performance-dashboard\outputs\villages_dashboard.gpkg


In [1]:
# Normally, at this point, I would use arcpy to publish the layer to AGOL as a hosted feature layer. However, because these were all saved as GeoPackages, and publishing those would require converting all of them to SEDFs first, I am instead opting to publish directly using the AGOL web interface.

# Which means that we have officially concluded the compilation process, and as a result, the Python portion of this project. But first:

print(f"*mic drop* Rawal out.")

*mic drop* Rawal out.
